# s24 — Git Worktree Isolation

**What this teaches:** how the agent can spin up an isolated git worktree, work in it, and merge the result back. Worktrees give per-task sandboxes so the main checkout stays clean even when the agent is editing files.

**Concept anchor:** a `git worktree` is a second working tree attached to the same `.git` directory. The agent's `worktree_create` / `worktree_remove` / `worktree_merge` tools wrap the plumbing so an LLM can manage them safely — without learning the full `git worktree` CLI surface.

## Prerequisites

- LLM provider configured. Default is `openai_compat` for self-hosted Ollama / vLLM — see [docs/providers.md](../../docs/providers.md).
- The notebook must run from inside a git repository (your Yoke checkout works).
- This walkthrough only **describes** the tools — no worktree is created. See [s25_conflicts](../s25_conflicts/s25_conflicts.ipynb) for an end-to-end create-and-merge demo.

## 1. Imports

`internal/worktree.Tools(repo)` builds the three tools scoped to a specific repository root.

In [ ]:
import (
    "context"
    "fmt"
    "os"

    "github.com/blouargant/yoke/core/agentkit"
    "github.com/blouargant/yoke/core/stream"
    "github.com/blouargant/yoke/internal/worktree"
)

## 2. Helper

Notebook-safe `must` (panic, not `os.Exit`).

In [ ]:
func must(err error) {
    if err != nil {
        panic(fmt.Sprintf("%v", err))
    }
}

## 3. Locate the repo and build the model

Worktree tools are bound to a single repo path at construction time, so it's clear *which* `.git` directory they will touch.

In [ ]:
ctx := context.Background()
llm, err := agentkit.NewModel(ctx)
must(err)

repo, _ := os.Getwd()
fmt.Println("repo:", repo)

## 4. Build the agent with worktree tools

The `Instruction` deliberately tells the model to *describe* the tools rather than *use* them — useful for documentation runs and for confirming what's available before granting it action permissions.

In [ ]:
a, err := agentkit.New(agentkit.AgentConfig{
    Name:        "s24_worktree",
    Description: "Worktree-aware agent.",
    Instruction: "Explain to the user what worktree_create/remove/merge do; do not execute them unless explicitly asked.",
    Model:       llm,
    Tools:       worktree.Tools(repo),
})
must(err)
r, err := agentkit.Runner("s24", a)
must(err)

## 5. Run

The agent should describe each of the three tools without actually creating a worktree.

In [ ]:
prompt := "Describe the three worktree tools you have and when to use each."
must(stream.Print(os.Stdout, agentkit.RunOnce(ctx, r, prompt)))

## What to look for

- The model produces a description but doesn't call any tool — the instruction is doing the work. Remove or relax it and the model will reach for `worktree_create`.
- See [s25_conflicts](../s25_conflicts/s25_conflicts.ipynb) for the full create → conflict-on-merge story.
- Combine with [s19_permissions](../s19_permissions/s19_permissions.ipynb) to require user approval before merges.

## Try it yourself

1. Change the prompt to "Create a worktree named `experiment` and report its path" — observe the model now calls `worktree_create`.
2. After creating one, ask: "Remove the experiment worktree" — the model calls `worktree_remove`.